# Neural Surface Reconstruction — Surface Constraints (PyTorch 튜토리얼)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonyoon-3dai/surface/blob/main/surface_constraints_pytorch.ipynb)

이 노트북은 같은 repo의 [`README.md`](./README.md)를 **PyTorch 코드 + 수식 + 시각화**로 라인 바이 라인 따라가는 튜토리얼입니다.

다루는 내용:
1. **SDF와 zero-level set**: 구(sphere)의 해석적 SDF를 만들고 등고선으로 시각화.
2. **Eikonal 제약**: $\|\nabla f\|_2=1$ 이 무엇이며 왜 필요한지, MLP가 자동으로 만족하지 않음을 확인.
3. **Surface-zero 제약**: 포인트 클라우드에 MLP로 SDF를 피팅하고 normal alignment까지.
4. **Unbiased volume rendering (NeuS)**: sigmoid $\Phi_s$ 와 opaque density로부터 weight $w(t)$를 계산, unbiased + occlusion-aware 성질 확인.
5. **VolSDF**: Laplace CDF를 사용한 대안 공식과 NeuS와의 비교.
6. **Photometric loss**: 토이 2D 장면에서 렌더링 색을 이미지와 비교.
7. **전체 loss**: 모든 항을 합친 학습 루프.
8. **NeRF와의 대비**: density 기반 표면 추출이 왜 어려운지 시각화.

Colab에서 바로 실행 가능합니다. 모든 학습 루프는 CPU로도 수 초 내에 끝나도록 작게 잡았습니다.


## 0. 환경 세팅

`torch`, `numpy`, `matplotlib` 만 사용합니다. Colab에는 모두 기본 설치되어 있습니다.

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch :', torch.__version__)


## 1. 기본 표현: SDF의 Zero-Level Set

표면 $\mathcal{S}$는 MLP로 parametrize된 implicit function $f_\theta: \mathbb{R}^3 \to \mathbb{R}$의 zero-level set입니다:

$$\mathcal{S} = \{\mathbf{x} \in \mathbb{R}^3 \mid f_\theta(\mathbf{x}) = 0\}$$

- $f_\theta(\mathbf{x}) < 0$ : 점 $\mathbf{x}$가 **내부**
- $f_\theta(\mathbf{x}) = 0$ : 점이 **표면 위**
- $f_\theta(\mathbf{x}) > 0$ : 점이 **외부**
- $|f_\theta(\mathbf{x})|$ : 표면까지의 거리

### 해석적 예제: 구(sphere)의 SDF

중심 $\mathbf{c}$, 반지름 $r$인 구의 SDF는 매우 간단합니다:

$$f_{\text{sphere}}(\mathbf{x}) = \|\mathbf{x} - \mathbf{c}\|_2 - r$$

이 함수는 $\|\nabla f\|_2 = 1$ 을 **정확히** 만족하는 이상적인 SDF입니다 (구의 바깥 거의 모든 점에서 $\nabla f = (\mathbf{x}-\mathbf{c})/\|\mathbf{x}-\mathbf{c}\|$).

2D 단면에서 시각화해봅시다.

In [ ]:
def sphere_sdf(x, c=torch.zeros(2), r=1.0):
    # x: (..., 2) 또는 (..., 3), 마지막 축이 좌표
    return torch.linalg.norm(x - c, dim=-1) - r

# 2D 평면에 SDF 값을 그리기
grid_n = 200
xs = torch.linspace(-2, 2, grid_n)
ys = torch.linspace(-2, 2, grid_n)
X, Y = torch.meshgrid(xs, ys, indexing='xy')
pts = torch.stack([X, Y], dim=-1)  # (n, n, 2)

sdf_vals = sphere_sdf(pts, c=torch.zeros(2), r=1.0).numpy()

fig, ax = plt.subplots(1, 2, figsize=(11, 5))

im = ax[0].imshow(sdf_vals, extent=[-2, 2, -2, 2], origin='lower', cmap='RdBu_r', vmin=-2, vmax=2)
ax[0].set_title(r'SDF $f(\mathbf{x}) = \|\mathbf{x}\| - 1$')
ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
plt.colorbar(im, ax=ax[0], shrink=0.8)

cs = ax[1].contour(X.numpy(), Y.numpy(), sdf_vals, levels=np.linspace(-1.5, 1.5, 13), cmap='RdBu_r')
ax[1].clabel(cs, fmt='%.1f', fontsize=8)
# zero-level set 강조
ax[1].contour(X.numpy(), Y.numpy(), sdf_vals, levels=[0.0], colors='black', linewidths=3)
ax[1].set_title('등고선 (zero-level set = 검정 굵은 선)')
ax[1].set_xlabel('x'); ax[1].set_ylabel('y'); ax[1].set_aspect('equal')

plt.tight_layout(); plt.show()

print('내부(-0.5, 0.0):', sphere_sdf(torch.tensor([-0.5, 0.0])).item(), '  < 0이면 내부')
print('표면(1.0, 0.0) :', sphere_sdf(torch.tensor([ 1.0, 0.0])).item(), '  ≈ 0 이면 표면 위')
print('외부(1.5, 0.0) :', sphere_sdf(torch.tensor([ 1.5, 0.0])).item(), '  > 0이면 외부')


## 2.1 Eikonal Constraint — 기울기 크기 = 1

SDF가 갖춰야 할 가장 기본적인 수학적 성질:

$$\|\nabla f(\mathbf{x})\|_2 = 1, \quad \forall \mathbf{x} \in \Omega$$

직관: 한 발짝($\Delta\mathbf{x}$) 움직이면 거리 값도 딱 그만큼 변해야 함. 그렇지 않으면 "거리 함수"라 부를 수 없음.

MLP $f_\theta$는 자동으로 이 조건을 만족하지 않기 때문에 **Eikonal regularizer** (Gropp et al., ICML 2020)를 도입합니다:

$$\mathcal{L}_{\text{eik}} = \frac{1}{N} \sum_{i=1}^{N} \left( \|\nabla f_\theta(\mathbf{x}_i)\|_2 - 1 \right)^2$$

먼저 해석적 sphere SDF는 이 조건을 자동으로 만족하는지 확인하고, 랜덤 초기화된 MLP는 얼마나 만족 못 하는지 봅시다.

In [ ]:
def grad_norm(f, x):
    """x에서 f의 grad의 L2 norm을 autograd로 계산."""
    x = x.clone().requires_grad_(True)
    y = f(x)
    # y가 scalar가 아닌 batch일 때는 sum()으로 backward
    g, = torch.autograd.grad(y.sum(), x, create_graph=True)
    return g.norm(dim=-1)

# (1) 해석적 sphere SDF의 grad norm
pts2 = torch.stack([X, Y], dim=-1).reshape(-1, 2)
gn_sphere = grad_norm(lambda x: sphere_sdf(x, c=torch.zeros(2), r=1.0), pts2)
print(f'[sphere SDF]   mean |grad| = {gn_sphere.mean().item():.4f}  (이상값 1.0)')
print(f'                 std       = {gn_sphere.std().item():.4f}')

# (2) 랜덤 초기화 MLP
class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden=64, n_layers=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden), nn.Softplus(beta=100)]
        for _ in range(n_layers - 2):
            layers += [nn.Linear(hidden, hidden), nn.Softplus(beta=100)]
        layers += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

torch.manual_seed(7)
mlp_rand = MLP()
gn_mlp = grad_norm(mlp_rand, pts2)
print(f'[random MLP ]   mean |grad| = {gn_mlp.mean().item():.4f}  (1.0과 거리 멈)')
print(f'                 std       = {gn_mlp.std().item():.4f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = ax[0].imshow(gn_sphere.detach().numpy().reshape(grid_n, grid_n),
                   extent=[-2, 2, -2, 2], origin='lower', cmap='viridis', vmin=0, vmax=2)
ax[0].set_title(r'해석적 SDF: $\|\nabla f\|$  ≈ 1 (이상)')
plt.colorbar(im0, ax=ax[0], shrink=0.8)

im1 = ax[1].imshow(gn_mlp.detach().numpy().reshape(grid_n, grid_n),
                   extent=[-2, 2, -2, 2], origin='lower', cmap='viridis', vmin=0, vmax=2)
ax[1].set_title(r'랜덤 MLP: $\|\nabla f\|$ (Eikonal 위반)')
plt.colorbar(im1, ax=ax[1], shrink=0.8)
plt.tight_layout(); plt.show()


### Eikonal loss 유무 비교 — 간단한 2D 학습

다음 실험:
- 표면: 원 (반지름 1) 위의 샘플들 — $f(\mathbf{x}_i) \approx 0$ 강제
- **실험 A (Eikonal 없음)**: surface loss만 → 학습 후 SDF가 아닌 "아무 함수"로 수렴할 수 있음
- **실험 B (Eikonal 있음)**: surface loss + eikonal loss → 진짜 SDF에 가까워짐

학습 후 `|grad|` 히스토그램과 zero-level set을 비교합니다.

In [ ]:
def train_2d(use_eikonal: bool, steps=1500, lr=1e-3):
    torch.manual_seed(42)
    net = MLP().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)

    # 표면 샘플 (단위 원) + ambient 샘플 (정규화용)
    def sample_surface(n):
        theta = 2 * math.pi * torch.rand(n, device=device)
        return torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)
    def sample_ambient(n):
        return (torch.rand(n, 2, device=device) * 4 - 2)  # [-2, 2]^2

    for step in range(steps):
        xs = sample_surface(512)
        xa = sample_ambient(1024).requires_grad_(True)

        loss_surf = net(xs).abs().mean()
        loss = loss_surf
        if use_eikonal:
            y = net(xa)
            g, = torch.autograd.grad(y.sum(), xa, create_graph=True)
            loss_eik = ((g.norm(dim=-1) - 1.0) ** 2).mean()
            loss = loss_surf + 0.1 * loss_eik

        opt.zero_grad(); loss.backward(); opt.step()
    return net

net_noeik = train_2d(False)
net_eik   = train_2d(True)

# Grid evaluation
grid = pts2.to(device)
def eval_field(net):
    net.eval()
    with torch.no_grad():
        return net(grid).reshape(grid_n, grid_n).cpu().numpy()

f_noeik = eval_field(net_noeik)
f_eik   = eval_field(net_eik)

# Grad norm histogram
def grad_norms_on_grid(net):
    g = grad_norm(net, grid)
    return g.detach().cpu().numpy()

gn_noeik = grad_norms_on_grid(net_noeik)
gn_eik   = grad_norms_on_grid(net_eik)

fig, ax = plt.subplots(2, 2, figsize=(11, 9))

for axi, f, title in [(ax[0,0], f_noeik, 'A. Eikonal 없음: level sets'),
                      (ax[0,1], f_eik,   'B. Eikonal 있음: level sets')]:
    cs = axi.contour(X.numpy(), Y.numpy(), f, levels=np.linspace(-1.5, 1.5, 13), cmap='RdBu_r')
    axi.clabel(cs, fmt='%.1f', fontsize=7)
    axi.contour(X.numpy(), Y.numpy(), f, levels=[0], colors='black', linewidths=3)
    # ground-truth circle
    theta = np.linspace(0, 2*np.pi, 200)
    axi.plot(np.cos(theta), np.sin(theta), '--', color='lime', lw=1.5, label='GT circle')
    axi.set_title(title); axi.set_aspect('equal'); axi.legend(loc='upper right', fontsize=8)

ax[1,0].hist(gn_noeik, bins=60, color='#d95f5f'); ax[1,0].axvline(1.0, color='k', ls='--')
ax[1,0].set_title(r'A. $\|\nabla f\|$ 분포 — 1 근처에 몰리지 않음')
ax[1,0].set_xlabel(r'$\|\nabla f\|$')

ax[1,1].hist(gn_eik, bins=60, color='#4a90d9'); ax[1,1].axvline(1.0, color='k', ls='--')
ax[1,1].set_title(r'B. $\|\nabla f\|$ 분포 — 1 근처에 집중')
ax[1,1].set_xlabel(r'$\|\nabla f\|$')

plt.tight_layout(); plt.show()

print(f'A (no eikonal): mean |grad| = {gn_noeik.mean():.3f}, std = {gn_noeik.std():.3f}')
print(f'B (eikonal)   : mean |grad| = {gn_eik.mean():.3f}, std = {gn_eik.std():.3f}')


**관찰:**
- A (Eikonal 없음): 표면 위에서 $f \approx 0$은 맞지만, level set 간격이 불균일하고 기울기 크기가 1에서 멀어짐.
- B (Eikonal 있음): level set이 등간격에 가깝고 기울기 크기가 1 주변에 집중 → 진짜 **signed distance**처럼 행동.

> README의 지적대로 Eikonal은 **필요조건**일 뿐 충분조건은 아닙니다. StEik, Viscosity Regularization 등이 추가로 제안된 이유입니다.

---

## 2.2 Surface-Zero Constraint + Normal Alignment

포인트 클라우드 $\{(\mathbf{x}_i, \mathbf{n}_i)\}$가 주어진 경우:

$$\mathcal{L}_{\text{surf}} = \underbrace{\sum_i |f_\theta(\mathbf{x}_i)|}_{\text{manifold}} + \lambda_n \underbrace{\sum_i \left(1 - \langle \nabla f_\theta(\mathbf{x}_i), \mathbf{n}_i \rangle \right)}_{\text{normal alignment}}$$

- 첫 항: 표면 위에서 $f=0$
- 둘째 항: 표면 법선 방향이 grad 방향과 일치 (cosine 1)

아래는 별 모양 2D 포인트 클라우드를 MLP SDF로 피팅하는 예제입니다.

In [ ]:
# 별 모양 윤곽선 샘플링 (5각 별의 부드러운 변형)
def sample_star(n):
    t = 2*math.pi*torch.rand(n)
    r = 1.0 + 0.35*torch.cos(5*t)
    x = torch.stack([r*torch.cos(t), r*torch.sin(t)], dim=-1)
    # tangent 방향: d(r cos t, r sin t)/dt, normal = tangent를 90도 회전
    dr = -0.35*5*torch.sin(5*t)
    tx = dr*torch.cos(t) - r*torch.sin(t)
    ty = dr*torch.sin(t) + r*torch.cos(t)
    tang = torch.stack([tx, ty], dim=-1)
    tang = tang / tang.norm(dim=-1, keepdim=True)
    # 바깥 방향 normal: (tang_y, -tang_x) 이거나 그 반대. 중심 기준으로 바깥쪽 선택
    nrm = torch.stack([tang[:,1], -tang[:,0]], dim=-1)
    # 중심에서 멀어지는 방향으로 부호 정렬
    sign = torch.sign((nrm * x).sum(-1, keepdim=True))
    nrm = nrm * sign
    return x, nrm

x_surf, n_surf = sample_star(400)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x_surf[:,0], x_surf[:,1], s=6, c='k')
for i in range(0, 400, 12):
    ax.arrow(x_surf[i,0].item(), x_surf[i,1].item(),
             0.15*n_surf[i,0].item(), 0.15*n_surf[i,1].item(),
             head_width=0.04, color='red', length_includes_head=True)
ax.set_title('별 모양 point cloud + 법선 (빨강)')
ax.set_aspect('equal'); ax.set_xlim(-2,2); ax.set_ylim(-2,2); plt.show()


In [ ]:
torch.manual_seed(0)
net_star = MLP(hidden=128, n_layers=5).to(device)
opt = torch.optim.Adam(net_star.parameters(), lr=1e-3)

x_surf_d = x_surf.to(device); n_surf_d = n_surf.to(device)

losses = []
for step in range(2500):
    xs = x_surf_d[torch.randint(0, x_surf_d.shape[0], (256,))]
    ns = n_surf_d[torch.randint(0, n_surf_d.shape[0], (256,))]
    xs = xs.requires_grad_(True)
    xa = (torch.rand(512, 2, device=device)*4 - 2).requires_grad_(True)

    fs = net_star(xs)
    fa = net_star(xa)
    gs, = torch.autograd.grad(fs.sum(), xs, create_graph=True)
    ga, = torch.autograd.grad(fa.sum(), xa, create_graph=True)

    loss_manifold  = fs.abs().mean()
    loss_normal    = (1 - (gs * ns).sum(-1)).mean()
    loss_eik       = ((ga.norm(dim=-1) - 1.0)**2).mean()
    loss_offsurf   = torch.exp(-100 * fa.abs()).mean()  # IGR의 'off-surface' 보조항

    loss = loss_manifold + 0.5*loss_normal + 0.1*loss_eik + 0.1*loss_offsurf
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(losses); ax[0].set_yscale('log'); ax[0].set_title('total loss'); ax[0].set_xlabel('step')

f_star = eval_field(net_star)
cs = ax[1].contourf(X.numpy(), Y.numpy(), f_star, levels=30, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
ax[1].contour(X.numpy(), Y.numpy(), f_star, levels=[0], colors='black', linewidths=3)
ax[1].scatter(x_surf[:,0], x_surf[:,1], s=4, c='lime')
ax[1].set_title('학습된 SDF (검정: zero-level set, 초록: GT point)')
ax[1].set_aspect('equal'); plt.colorbar(cs, ax=ax[1], shrink=0.8)
plt.tight_layout(); plt.show()


## 2.3 Unbiased & Occlusion-aware Volume Rendering

NeuS 논문의 핵심 질문: **SDF를 volume rendering weight $w(t)$로 변환할 때, weight가 "표면 위에서 최대"가 되어야 하고(unbiased), 동시에 앞 표면이 뒤 표면을 가려야 한다(occlusion-aware).**

Ray: $\mathbf{r}(t) = \mathbf{o} + t\mathbf{v}$, 색:

$$C(\mathbf{r}) = \int_0^\infty w(t)\, c(\mathbf{r}(t), \mathbf{v})\, dt$$

### NeuS 공식

$\Phi_s(x) = (1+e^{-sx})^{-1}$ 를 sigmoid라 하면 **opaque density**:

$$\rho(t) = \max\!\left( \frac{-\frac{d\Phi_s}{dt}(f(\mathbf{r}(t)))}{\Phi_s(f(\mathbf{r}(t)))},\ 0 \right)$$

Discrete weight:

$$w_i = T_i\,\alpha_i,\quad T_i = \prod_{j<i}(1-\alpha_j),\quad
\alpha_i = \max\!\left(\frac{\Phi_s(f(\mathbf{x}_i)) - \Phi_s(f(\mathbf{x}_{i+1}))}{\Phi_s(f(\mathbf{x}_i))},\ 0\right)$$

먼저 $\Phi_s$ 를 $s$ 변화에 따라 그려봅시다. $s$가 커질수록 표면이 날카로워집니다.


In [ ]:
def Phi_s(x, s):
    return torch.sigmoid(s * x)

x_axis = torch.linspace(-2, 2, 400)
plt.figure(figsize=(7, 4.5))
for s in [1, 4, 16, 64]:
    plt.plot(x_axis, Phi_s(x_axis, s), label=f's = {s}')
plt.axvline(0, color='k', ls='--', lw=0.7)
plt.title(r'$\Phi_s(x) = 1/(1+e^{-sx})$   —   $s$ 증가 → step-like')
plt.xlabel('x (signed distance)'); plt.ylabel(r'$\Phi_s(x)$')
plt.legend(); plt.grid(alpha=0.3); plt.show()


### 단일 표면 ray: NeuS weight가 표면에서 최대임을 확인 (unbiased)

Ray 위의 SDF를 $f(t)$로 두겠습니다. 표면이 $t^\* = 3$에 있고 ray 방향이 표면 법선과 평행할 때, $f(t) = t^\* - t$ (내부로 들어갈수록 음수) 형태의 선형 모델이 성립합니다 (NeuS 증명의 가정).

In [ ]:
def neus_weights(f_vals, s):
    """f_vals: (T,) — ray 위 각 샘플의 SDF 값. s: scalar."""
    phi = torch.sigmoid(s * f_vals)
    # alpha_i = max((phi_i - phi_{i+1})/phi_i, 0), 마지막은 0
    phi_next = torch.cat([phi[1:], phi[-1:]])
    alpha = torch.clamp((phi - phi_next) / (phi + 1e-12), min=0.0)
    alpha[-1] = 0.0
    # T_i = prod_{j<i} (1 - alpha_j)
    trans = torch.cumprod(torch.cat([torch.ones(1), 1 - alpha[:-1]]), dim=0)
    w = trans * alpha
    return w, alpha, trans

# Ray: t in [0, 6], 표면 t* = 3, f(t) = 3 - t (ray 방향 = 안쪽 바라봄)
T_samples = 256
t = torch.linspace(0, 6, T_samples)
t_star = 3.0
f_ray = t_star - t   # t < t* → f>0 (외부), t > t* → f<0 (내부)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(t, f_ray); ax[0].axhline(0, color='k', ls='--'); ax[0].axvline(t_star, color='r', ls=':')
ax[0].set_title(r'ray 위 SDF $f(t) = t^*-t$'); ax[0].set_xlabel('t'); ax[0].set_ylabel('f')

for s in [2, 8, 32]:
    w, _, _ = neus_weights(f_ray, s)
    ax[1].plot(t, w.numpy(), label=f's={s}')
ax[1].axvline(t_star, color='r', ls=':', label=r'$t^*$ (표면)')
ax[1].set_title(r'NeuS weight $w(t)$'); ax[1].set_xlabel('t'); ax[1].legend(); ax[1].grid(alpha=0.3)

# 비교용: 순진한 α = 1 - Φ(f)와 같은 '잘못된' 스킴
def naive_weights(f_vals, s):
    phi = torch.sigmoid(s * f_vals)
    alpha = torch.clamp(1 - phi, min=0.0)
    trans = torch.cumprod(torch.cat([torch.ones(1), 1 - alpha[:-1]]), dim=0)
    return trans * alpha

for s in [2, 8, 32]:
    w_bad = naive_weights(f_ray, s)
    ax[2].plot(t, w_bad.numpy(), label=f's={s}')
ax[2].axvline(t_star, color='r', ls=':', label=r'$t^*$')
ax[2].set_title(r'순진한 $w(t)$ (biased) — 최대가 $t^*$가 아님')
ax[2].set_xlabel('t'); ax[2].legend(); ax[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# argmax 위치 확인
for s in [2, 8, 32]:
    w, _, _ = neus_weights(f_ray, s)
    print(f's={s:>2}: NeuS argmax t = {t[w.argmax()].item():.3f}  (GT t*={t_star})')


### 두 개의 표면 ray: Occlusion-aware

Ray가 두 개의 표면을 순서대로 관통한다고 합시다. 앞 표면을 $t_1=2$, 뒤 표면을 $t_2=4$. NeuS의 $T_i$ (transmittance)가 앞 표면 이후 급격히 감소해서, 뒤 표면의 weight가 작아져야 합니다.

In [ ]:
# 두 표면: f가 0을 두 번 교차. f(t) = -(t-t1)(t-t2)/(t2-t1) * 축소 — 간단히 0번 근처에서 선형으로.
# 여기선 간단하게 |t - 2| - 0.3 같은 작은 조각들을 이어붙이거나,
# 더 간단히 실제 ray의 SDF: 두 구의 합성을 흉내내어 부호가 + - + - + 패턴을 만든다.
t = torch.linspace(0, 6, 512)
# 바깥(+) → 첫 번째 쉘 안(-) → 사이 공간(+) → 두 번째 쉘 안(-) → 바깥(+)
# 직관적으로: 원래 구 1: |t-2|-0.4, 구 2: |t-4|-0.4, 전체 SDF = min(둘) (but sign은 유지)
# 더 쉽게: 삼각파를 쓴다.
def two_surface_sdf(t):
    t1, t2, r = 2.0, 4.0, 0.4
    # 구 내부 두께 ~0.5
    d1 = torch.abs(t - t1) - r
    d2 = torch.abs(t - t2) - r
    # 두 구의 합집합의 SDF는 min. 여기서 부호가 두 군데에서 음수가 됨.
    return torch.minimum(d1, d2)

f_ray = two_surface_sdf(t)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(t, f_ray); ax[0].axhline(0, color='k', ls='--')
for s_, lab in [(2.0, 't1'), (4.0, 't2')]:
    ax[0].axvline(s_, color='r', ls=':', alpha=0.7)
ax[0].set_title('ray 위 SDF: 두 개의 표면'); ax[0].set_xlabel('t')

# 중요: NeuS에서는 단조 감소 가정이 깨질 때 max(·,0)이 작용. 그대로 적용.
for s in [8, 32]:
    w, alpha, trans = neus_weights(f_ray, s)
    ax[1].plot(t, w.numpy(), label=f's={s}')
ax[1].axvline(2, color='r', ls=':'); ax[1].axvline(4, color='orange', ls=':')
ax[1].set_title('NeuS weight $w(t)$ — 두 표면'); ax[1].set_xlabel('t'); ax[1].legend(); ax[1].grid(alpha=0.3)

# transmittance
for s in [8, 32]:
    _, _, trans = neus_weights(f_ray, s)
    ax[2].plot(t, trans.numpy(), label=f's={s}')
ax[2].axvline(2, color='r', ls=':'); ax[2].axvline(4, color='orange', ls=':')
ax[2].set_title(r'Transmittance $T(t)$ — 앞 표면 통과 후 급감')
ax[2].set_xlabel('t'); ax[2].legend(); ax[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

for s in [8, 32]:
    w, _, _ = neus_weights(f_ray, s)
    # 두 봉우리 중 첫 번째가 더 커야 occlusion-aware
    # 근사적으로 mid-point로 분할
    mask1 = t < 3
    mask2 = t >= 3
    print(f's={s}: 앞 표면 weight 최대 = {w[mask1].max():.4f}, '
          f'뒤 표면 weight 최대 = {w[mask2].max():.4f}  → 앞 > 뒤 (occlusion-aware)')


### VolSDF: Laplace CDF를 이용한 대안

VolSDF는 SDF→density 변환에 **Laplace CDF**를 사용합니다:

$$\sigma_{\text{VolSDF}}(t) = \alpha \cdot h_{\text{Laplace}}(-s f(t))$$

$$h_{\text{Laplace}}(-sf) = \begin{cases} \tfrac{1}{2}\exp(-sf), & f > 0 \\ 1 - \tfrac{1}{2}\exp(sf), & f \leq 0 \end{cases}$$

UNIS (Deng et al., ICCV 2025) 분석: $\alpha = -2 s f'(t)$ 로 두면 NeuS와 동일한 first-order unbiasedness를 얻습니다:

$$\sigma_{\text{VolSDF-unbiased}}(t) = -2sf'(t)\, h_{\text{Laplace}}(-sf(t))$$

NeuS weight와 VolSDF weight를 같은 ray에서 그려 비교해봅시다.

In [ ]:
def laplace_h(z):
    """h_Laplace(z)로 정의가 일관되도록 z = -s f 를 받음."""
    return torch.where(z < 0, 0.5 * torch.exp(z), 1 - 0.5 * torch.exp(-z))

def volsdf_density(f_vals, s, alpha=None, dt=None):
    z = -s * f_vals
    h = laplace_h(z)
    if alpha is None:
        alpha = 1.0
    return alpha * h

def volsdf_weights(sigma, dt):
    """sigma (T,) → w_i = T_i * (1 - exp(-sigma_i dt))"""
    alpha = 1 - torch.exp(-sigma * dt)
    trans = torch.cumprod(torch.cat([torch.ones(1), 1 - alpha[:-1]]), dim=0)
    return trans * alpha

# 단일 표면 다시 (t* = 3, f = 3 - t)
t = torch.linspace(0, 6, 512)
dt = (t[1] - t[0]).item()
f_ray = 3.0 - t

s = 16.0

# 1) Plain VolSDF (alpha = 1)  — biased in general but commonly used
sig_plain = volsdf_density(f_ray, s, alpha=1.0)
w_vol_plain = volsdf_weights(sig_plain, dt)

# 2) UNIS-corrected VolSDF: alpha(t) = -2 s f'(t); f' = -1 here → alpha = 2s
fprime = -1.0  # f = 3 - t
alpha_t = -2 * s * fprime  # scalar here
sig_corr = volsdf_density(f_ray, s, alpha=alpha_t)
w_vol_corr = volsdf_weights(sig_corr, dt)

# 3) NeuS (정규화된 weight — 이산 공식)
w_neus, _, _ = neus_weights(f_ray, s)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(t, sig_plain, label=r'$\sigma$ VolSDF ($\alpha=1$)')
ax[0].plot(t, sig_corr,  label=r'$\sigma$ UNIS ($\alpha=-2sf\'$)', ls='--')
ax[0].axvline(3.0, color='r', ls=':')
ax[0].set_title('SDF → density'); ax[0].set_xlabel('t'); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(t, w_neus/w_neus.max(),     label='NeuS (norm)')
ax[1].plot(t, w_vol_plain/w_vol_plain.max(), label='VolSDF plain (norm)', ls='--')
ax[1].plot(t, w_vol_corr/w_vol_corr.max(),   label='VolSDF UNIS (norm)', ls=':')
ax[1].axvline(3.0, color='r', ls=':')
ax[1].set_title(r'weight $w(t)$ (정규화 비교)'); ax[1].set_xlabel('t'); ax[1].legend(); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

for name, w in [('NeuS', w_neus), ('VolSDF-plain', w_vol_plain), ('VolSDF-UNIS', w_vol_corr)]:
    print(f'{name:>14s}: argmax t = {t[w.argmax()].item():.3f}  (GT 3.000)')


## 2.4 Photometric Constraint — 이미지 일관성

학습 신호는 최종적으로 렌더 이미지 $\hat{C}(\mathbf{r})$ 과 GT 이미지 $C(\mathbf{r})$ 의 L1 오차에서 옵니다:

$$\mathcal{L}_{\text{rgb}} = \frac{1}{|\mathcal{R}|} \sum_{\mathbf{r}} \left\| \hat{C}(\mathbf{r}) - C(\mathbf{r}) \right\|_1$$

토이 예제: ray 샘플 하나를 만들고, 앞에서 만든 NeuS weight와 ray 위의 색 $c(t)$를 합쳐 최종 색을 적분해봅시다.

In [ ]:
# 두 표면 ray 재사용. 각 표면에 색이 있다고 가정: 앞=red (1,0,0), 뒤=blue (0,0,1)
t = torch.linspace(0, 6, 512)
f_ray = two_surface_sdf(t)
s = 32.0
w, _, _ = neus_weights(f_ray, s)

# 색: 앞 표면 근처 red, 뒤 표면 근처 blue (Gaussian 가중)
c_red  = torch.tensor([1.0, 0.0, 0.0])
c_blue = torch.tensor([0.0, 0.0, 1.0])
wr = torch.exp(-((t - 2.0)/0.3)**2).unsqueeze(-1)
wb = torch.exp(-((t - 4.0)/0.3)**2).unsqueeze(-1)
c_t = wr * c_red + wb * c_blue  # (T, 3)

# 렌더 색: sum_i w_i c_i  (정규화 안 함 — background alpha는 무시)
C_hat = (w.unsqueeze(-1) * c_t).sum(0)
print(f'렌더 색 C_hat = {C_hat.numpy()}  (앞 표면 red가 지배 → occlusion-aware)')

# GT 색을 앞 표면 red로 가정하고 L1 loss
C_gt = torch.tensor([1.0, 0.0, 0.0])
L_rgb = (C_hat - C_gt).abs().sum()
print(f'L_rgb = {L_rgb.item():.4f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(t, w.numpy(), color='k'); ax[0].set_title(r'$w(t)$ (NeuS)')
ax[0].axvline(2, color='r', ls=':'); ax[0].axvline(4, color='b', ls=':')
ax[0].set_xlabel('t'); ax[0].grid(alpha=0.3)

ax[1].plot(t, c_t[:,0], color='r', label='R'); ax[1].plot(t, c_t[:,2], color='b', label='B')
ax[1].set_title(r'$c(t)$'); ax[1].set_xlabel('t'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 빨간 띠 표시
plt.figure(figsize=(4, 1.2))
plt.imshow(np.tile(C_hat.numpy().reshape(1,1,3), (1, 100, 1)))
plt.title(f'렌더 색 C_hat'); plt.axis('off'); plt.show()


## 3. 전체 Loss — 모든 항 합치기

$$\mathcal{L} = \mathcal{L}_{\text{rgb}} + \lambda_{\text{eik}} \mathcal{L}_{\text{eik}} + \lambda_{\text{mask}} \mathcal{L}_{\text{mask}} + \lambda_{\text{smooth}} \mathcal{L}_{\text{smooth}}$$

전형적: $\lambda_{\text{eik}} = 0.1$, $\lambda_{\text{mask}} = 0.1$.

아래는 "NeuS-style 학습 스텝"을 **한 함수로** 집약한 것입니다. 실제 NeuS는 여기에 hierarchical sampling, background NeRF 등이 더해집니다.

In [ ]:
class SDFNet(nn.Module):
    """3D SDF MLP (geometric initialization은 생략, 토이용)."""
    def __init__(self, hidden=128, n_layers=5):
        super().__init__()
        self.pe_L = 4
        in_dim = 3 + 3 * 2 * self.pe_L
        layers = []
        d = in_dim
        for _ in range(n_layers - 1):
            layers += [nn.Linear(d, hidden), nn.Softplus(beta=100)]
            d = hidden
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)

    def pe(self, x):
        freqs = 2 ** torch.arange(self.pe_L, device=x.device).float()
        xb = x.unsqueeze(-1) * freqs * math.pi
        return torch.cat([x, torch.sin(xb).flatten(-2), torch.cos(xb).flatten(-2)], dim=-1)

    def forward(self, x):
        return self.net(self.pe(x)).squeeze(-1)

class ColorNet(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3 + 3 + 3, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 3), nn.Sigmoid(),
        )
    def forward(self, x, n, v):
        return self.net(torch.cat([x, n, v], dim=-1))


def neus_step(sdf_net, color_net, rays_o, rays_v, rgb_gt, s, n_samples=64):
    """단일 학습 스텝: NeuS volume rendering + 모든 loss 계산."""
    B = rays_o.shape[0]
    t = torch.linspace(0.1, 3.0, n_samples, device=rays_o.device).expand(B, n_samples)
    pts = rays_o[:, None, :] + rays_v[:, None, :] * t[..., None]  # (B, T, 3)
    pts_flat = pts.reshape(-1, 3).requires_grad_(True)

    f = sdf_net(pts_flat)
    g, = torch.autograd.grad(f.sum(), pts_flat, create_graph=True)
    normals = g / (g.norm(dim=-1, keepdim=True) + 1e-8)

    f = f.view(B, n_samples)
    # NeuS weight (배치)
    phi = torch.sigmoid(s * f)
    phi_next = torch.cat([phi[:, 1:], phi[:, -1:]], dim=1)
    alpha = torch.clamp((phi - phi_next) / (phi + 1e-10), min=0.0)
    alpha[:, -1] = 0
    trans = torch.cumprod(torch.cat([torch.ones(B, 1, device=f.device), 1 - alpha[:, :-1]], dim=1), dim=1)
    w = trans * alpha                                                  # (B, T)

    v_rep = rays_v[:, None, :].expand(-1, n_samples, -1).reshape(-1, 3)
    colors = color_net(pts_flat.detach(), normals, v_rep).view(B, n_samples, 3)
    rgb_pred = (w[..., None] * colors).sum(1)

    # losses
    L_rgb = (rgb_pred - rgb_gt).abs().mean()
    L_eik = ((g.norm(dim=-1) - 1.0) ** 2).mean()
    L = L_rgb + 0.1 * L_eik
    return L, dict(rgb=L_rgb.item(), eik=L_eik.item(), rgb_pred=rgb_pred.detach())

print('학습 스텝 함수 정의 완료. (전체 훈련은 NeuS 레포 참고)')


In [ ]:
# 1-스텝 실행 예시: 가짜 ray 8개로 loss 값 확인
torch.manual_seed(0)
sdf_net = SDFNet().to(device)
col_net = ColorNet().to(device)
opt = torch.optim.Adam(list(sdf_net.parameters()) + list(col_net.parameters()), lr=5e-4)

rays_o = torch.randn(8, 3, device=device) * 0.1 + torch.tensor([0, 0, 2.5], device=device)
rays_v = torch.tensor([[0,0,-1.0]], device=device).expand(8, -1)
rgb_gt = torch.rand(8, 3, device=device)

for step in range(30):
    L, info = neus_step(sdf_net, col_net, rays_o, rays_v, rgb_gt, s=16.0)
    opt.zero_grad(); L.backward(); opt.step()
    if step % 5 == 0:
        print(f'step {step:2d}: L={L.item():.4f}  L_rgb={info["rgb"]:.4f}  L_eik={info["eik"]:.4f}')


## 4. 왜 Surface Constraint가 필수적인가 — NeRF와의 대비

NeRF는 density $\sigma(\mathbf{x})$ 자체를 배우지만 **level set에 대한 강한 제약이 없어서** 학습된 field에서 표면을 뽑기 어렵습니다. 어디서 $\sigma$ 값이 "임계값을 넘는다"로 표면을 정의할지 자체가 불분명합니다.

SDF + Eikonal + unbiased 렌더링 조합은:

- **표면 위치의 유일성**: $f=0$이 표면
- **기하학적 유효성**: $\|\nabla f\|=1$ 로 실제 거리처럼 행동
- **Occlusion 처리**: $T_i$ 가 자연스럽게 앞 면을 존중
- **abrupt depth change에 강건**: volume rendering의 장점 유지

아래 실험: 같은 2D 장면 (별 모양)에 대해 (A) density처럼 학습된 scalar field와 (B) SDF를 비교합니다. level set이 얼마나 다르게 추출되는지 봅시다.

In [ ]:
# (A) density MLP: 표면 근처에서 높은 값을 내도록 학습 (NeRF 스타일)
torch.manual_seed(0)
den_net = MLP(hidden=128, n_layers=5).to(device)
opt = torch.optim.Adam(den_net.parameters(), lr=1e-3)

for step in range(1500):
    xs = x_surf_d[torch.randint(0, x_surf_d.shape[0], (256,))]
    xa = (torch.rand(512, 2, device=device)*4 - 2)

    # density: 표면 근처는 +5, 멀리는 0 (softplus 이후 양수로)
    d_surf = den_net(xs)
    d_amb  = den_net(xa)
    loss = (d_surf - 5.0).pow(2).mean() + d_amb.pow(2).mean() * 0.01
    opt.zero_grad(); loss.backward(); opt.step()

den_field = eval_field(den_net)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im0 = ax[0].imshow(den_field, extent=[-2,2,-2,2], origin='lower', cmap='magma', vmin=-2, vmax=6)
ax[0].set_title('(A) Density-like field (NeRF 스타일)')
# 여러 임계값에서 level set을 그림 — '표면'이 어디인지 모호
for thr, col in [(1.0,'white'), (2.0,'cyan'), (4.0,'yellow')]:
    ax[0].contour(X.numpy(), Y.numpy(), den_field, levels=[thr], colors=col, linewidths=1.5)
ax[0].scatter(x_surf[:,0], x_surf[:,1], s=3, c='lime')
plt.colorbar(im0, ax=ax[0], shrink=0.8)
ax[0].set_aspect('equal')

im1 = ax[1].imshow(f_star, extent=[-2,2,-2,2], origin='lower', cmap='RdBu_r', vmin=-1.5, vmax=1.5)
ax[1].contour(X.numpy(), Y.numpy(), f_star, levels=[0], colors='black', linewidths=3)
ax[1].scatter(x_surf[:,0], x_surf[:,1], s=3, c='lime')
ax[1].set_title('(B) SDF: zero-level set이 명확')
plt.colorbar(im1, ax=ax[1], shrink=0.8)
ax[1].set_aspect('equal')

plt.tight_layout(); plt.show()
print('density 기반: 임계값(1, 2, 4)마다 다른 "표면"이 나옴 → 유일성 없음')
print('SDF 기반 : zero-level set 하나로 표면이 유일하게 정의됨')


## 5. 최신 동향 요약

- **NeuRodin** (Wang et al., NeurIPS 2024): NeuS/VolSDF 모두 SDF→density 변환에 bias가 있음을 지적. Eikonal loss가 과도한 smoothing을 유발한다고 분석하고, stochastic-step numerical gradient와 explicit bias correction을 제안.
- **UNIS** (Deng et al., ICCV 2025): VolSDF, NeuS, HF-NeuS를 특수 사례로 통합. 기존의 "학습 가능 파라미터"를 적절히 고르면 first-order unbiasedness가 자연스럽게 달성됨을 보임 — 위 `volsdf_weights` 실험에서 $\alpha = -2sf'$ 를 넣자 NeuS와 일치한 argmax를 얻은 것이 그 예시입니다.
- **StEik** (Yang et al., NeurIPS 2023): Eikonal loss의 continuum 한계에서 불안정성을 분석하고 directional divergence regularizer를 추가.

---

## 정리

이 노트북에서 다음을 코드로 재현했습니다:

| README 개념 | 노트북에서의 대응 |
|---|---|
| §1 SDF + zero-level set | `sphere_sdf`, 2D 등고선 + 검정색 0-선 |
| §2.1 Eikonal | `grad_norm`, 훈련 A/B 비교, `|grad|` 히스토그램 |
| §2.2 Surface-zero + normal | 별 포인트 클라우드 피팅 |
| §2.3 Unbiased & occlusion-aware | `neus_weights`, 단일/두 표면 ray |
| §2.3 VolSDF + UNIS | Laplace CDF, $\alpha=-2sf'$ 보정 |
| §2.4 Photometric | $w(t)$와 $c(t)$의 적분 |
| §3 전체 loss | `neus_step` 함수 |
| §4 NeRF 대비 | density vs SDF level set 시각화 |

각 섹션은 독립적으로 실행 가능하니, 관심 있는 부분만 골라 수정하며 실험해보세요.

### 추천 다음 단계

1. Positional encoding의 주파수 수를 바꿔 high-frequency detail을 관찰하기 (HF-NeuS 아이디어)
2. `s` 스케줄링: 학습 중 $s$를 늘려가며 sharpness 변화 관찰 (NeuS annealing)
3. 3D로 확장: 위 `SDFNet`에 real-world multi-view 데이터를 붙여 실제 reconstruction 파이프라인 구성 (원래 NeuS repo 참고)
